
# Revision 08 — K=46 edge-trim sensitivity for held-out top-5 spatial MS-AA

This notebook tests whether the sparse held-out result at the locked representative model order is disproportionately driven by run-boundary timepoints.

## Question

> Does the \(K=46\) held-out top-5 result survive when the first and last 10 TRs are excluded from **both archetype ranking and held-out timepoint identification**?

The MS-AA model is **not refit**. We use the same saved \(K=46\) across-condition spatial-AA solution and repeat only the downstream ranking/evaluation step after removing the boundary timepoints.

The trimmed reconstruction is re-standardized **after trimming**, so excluded timepoints do not influence the feature means or standard deviations used by the decoder.

## Locked settings

- spatial MS-AA
- across-condition fit
- \(K=46\)
- top-\(m=5\)
- 20 outer participant splits
- two cross-fit directions per outer split
- selected interpretation archetypes from the main analysis: **[4, 5, 6, 12, 8]**
- edge sensitivity: remove first 10 and last 10 TRs

## Outputs

1. Baseline vs edge-trimmed held-out top-5 accuracy.
2. Above-chance information retained after trimming.
3. Baseline vs trimmed top-5 archetype-selection frequency.
4. Jaccard overlap of the selected top-5 sets within each cross-fit.
5. Single-archetype ranking performance before vs after trimming for the five interpretation archetypes.

When the original held-out CSVs are present, the notebook reuses their **exact outer split seeds** so the trimmed analysis is paired to the original analysis. If they are unavailable, it generates a fresh set of splits and recomputes both baseline and trimmed analyses on those same splits.


In [ ]:

# ============================================================
# SETTINGS
# ============================================================

from pathlib import Path

ANALYSIS_TYPE = "spatial"
FIT_SCOPE = "across"

K = 46
TOP_M = 5
EDGE_TRIM = 10

LOCKED_ARCHETYPES = [4, 5, 6, 12, 8]

CONDITIONS = ["intact", "word", "rest"]

N_OUTER_REPS = 20
NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42

FIT_LOAD_DIR = Path("msaa_flexible_outputs_npz")
FIT_FILE = (
    FIT_LOAD_DIR
    / f"spatialAA_across_acrossCond_K{K}.npz"
)

# Original held-out analysis outputs.
BASELINE_DIR = Path(
    f"msaa_heldout_topm_outputs_{ANALYSIS_TYPE}_{FIT_SCOPE}"
)
BASELINE_RUN_CSV = (
    BASELINE_DIR
    / "heldout_topm_crossfit_runs.csv"
)
BASELINE_RANK_CSV = (
    BASELINE_DIR
    / "heldout_per_archetype_ranking_runs.csv"
)

OUTPUT_DIR = Path(
    "msaa_edge_trim_sensitivity_K46_top5"
)
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRIM_RUN_CSV = (
    OUTPUT_DIR
    / "edge_trim_top5_crossfit_runs.csv"
)
TRIM_RANK_CSV = (
    OUTPUT_DIR
    / "edge_trim_per_archetype_ranking_runs.csv"
)

print("Fit:", FIT_FILE)
print("K:", K)
print("top-m:", TOP_M)
print("Edge trim:", EDGE_TRIM, "TRs from each end")
print("Output:", OUTPUT_DIR.resolve())


In [ ]:

%matplotlib inline

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from IPython.display import display


## 1. Load the saved \(K=46\) fit

In [ ]:

def load_msaa_npz(path):

    data = np.load(
        path,
        allow_pickle=True,
    )

    results_subj = (
        data["results_subj"]
        .tolist()
    )

    if isinstance(
        results_subj,
        np.ndarray,
    ):
        results_subj = (
            results_subj.tolist()
        )

    out = {
        "K": (
            int(
                np.asarray(
                    data["K"]
                ).item()
            )
            if "K" in data
            else None
        ),
        "results_subj": results_subj,
    }

    for key in [
        "condition_labels_str",
        "condition_codes",
        "condition_names",
        "C",
        "analysis_type",
        "fit_scope",
    ]:
        if key in data:
            val = data[key]
            try:
                out[key] = val.tolist()
            except Exception:
                out[key] = val

    return out


if not FIT_FILE.exists():
    raise FileNotFoundError(
        f"Missing fit file: {FIT_FILE}"
    )

fit = load_msaa_npz(
    FIT_FILE
)

assert fit["K"] == K

condition_labels = np.asarray(
    fit["condition_labels_str"]
).astype(str)

print(
    "N fit entries:",
    len(
        fit["results_subj"]
    ),
)

print(
    "Condition counts:"
)

display(
    pd.Series(
        condition_labels
    )
    .value_counts()
    .reindex(
        CONDITIONS
    )
    .to_frame(
        "n"
    )
)

example = fit[
    "results_subj"
][0]

print(
    "sXC:",
    np.asarray(
        example["sXC"]
    ).shape,
)

print(
    "S:",
    np.asarray(
        example["S"]
    ).shape,
)


In [ ]:

def get_condition_subjects(
    fit,
    condition,
):

    labels = np.asarray(
        fit[
            "condition_labels_str"
        ]
    ).astype(str)

    idx = np.where(
        labels == condition
    )[0]

    return [
        fit[
            "results_subj"
        ][i]
        for i
        in idx
    ]



## 2. Reconstruction and decoding

The key sensitivity change occurs in `reconstruct_from_archetypes`:

1. reconstruct from the saved MS-AA solution;
2. remove the first/last `edge_trim` timepoints;
3. **then** Z-score each reconstructed node over the retained timepoints.

Thus the discarded boundary timepoints cannot influence either ranking, decoding, or post-reconstruction scaling.


In [ ]:

def to_float_array(x):
    return np.asarray(
        x,
        dtype=float,
    )


def reconstruct_from_archetypes(
    sub,
    analysis_type,
    archetype_indices=None,
    edge_trim=0,
):

    sXC = to_float_array(
        sub["sXC"]
    )

    S = to_float_array(
        sub["S"]
    )

    if archetype_indices is None:

        idx = np.arange(
            S.shape[0]
        )

    else:

        idx = np.asarray(
            list(
                archetype_indices
            ),
            dtype=int,
        )

    if analysis_type == "spatial":

        Xhat = (
            sXC[:, idx]
            @ S[idx, :]
        )

    elif analysis_type == "temporal":

        Xhat = (
            sXC[:, idx]
            @ S[idx, :]
        ).T

    else:

        raise ValueError(
            "analysis_type must be "
            "'spatial' or 'temporal'"
        )

    if edge_trim > 0:

        if (
            2 * edge_trim
            >= Xhat.shape[0]
        ):
            raise ValueError(
                "edge_trim removes all "
                "or nearly all timepoints."
            )

        Xhat = Xhat[
            edge_trim:-edge_trim,
            :
        ]

    # Standardize only over timepoints
    # that actually enter this analysis.
    Xhat = (
        Xhat
        - Xhat.mean(
            axis=0,
            keepdims=True,
        )
    ) / (
        Xhat.std(
            axis=0,
            keepdims=True,
        )
        + 1e-8
    )

    return Xhat


def build_recon_stack(
    subjects,
    analysis_type,
    archetype_indices=None,
    edge_trim=0,
):

    return np.stack(
        [
            reconstruct_from_archetypes(
                sub,
                analysis_type,
                archetype_indices=(
                    archetype_indices
                ),
                edge_trim=edge_trim,
            )
            for sub
            in subjects
        ],
        axis=0,
    )


def decoder(corrs):

    out = pd.DataFrame(
        {
            "rank": [0.0],
            "accuracy": [0.0],
            "error": [0.0],
        }
    )

    T = corrs.shape[0]

    for t in range(T):

        decoded_ind = int(
            np.argmax(
                corrs[t, :]
            )
        )

        out.loc[
            0,
            "error",
        ] += (
            np.mean(
                np.abs(
                    decoded_ind - t
                )
            )
            / T
        )

        out.loc[
            0,
            "accuracy",
        ] += (
            decoded_ind == t
        )

        out.loc[
            0,
            "rank",
        ] += np.mean(
            (
                corrs[
                    t,
                    :
                ]
                <= corrs[
                    t,
                    t
                ]
            ).astype(int)
        )

    out[
        "error"
    ] /= T

    out[
        "accuracy"
    ] /= T

    out[
        "rank"
    ] /= T

    return out


def get_xval_assignments(
    ndata,
    nfolds,
    rng=None,
):

    rng = (
        np.random.default_rng()
        if rng is None
        else rng
    )

    group_assignments = np.zeros(
        ndata,
        dtype=int,
    )

    groupsize = int(
        np.ceil(
            ndata
            / nfolds
        )
    )

    for i in range(
        1,
        nfolds,
    ):

        inds = np.arange(
            i * groupsize,
            min(
                (i + 1)
                * groupsize,
                ndata,
            ),
        )

        group_assignments[
            inds
        ] = i

    rng.shuffle(
        group_assignments
    )

    return group_assignments


def run_timepoint_decoding(
    recon_stack,
    nfolds=2,
    nreps=20,
    seed=42,
):

    N, T, V = (
        recon_stack.shape
    )

    rng = np.random.default_rng(
        seed
    )

    rows = []

    for rep in range(
        nreps
    ):

        fold_ids = (
            get_xval_assignments(
                N,
                nfolds,
                rng=rng,
            )
        )

        for fold in range(
            nfolds
        ):

            in_mask = (
                fold_ids
                == fold
            )

            out_mask = (
                ~in_mask
            )

            in_mean = (
                recon_stack[
                    in_mask
                ]
                .mean(
                    axis=0
                )
            )

            out_mean = (
                recon_stack[
                    out_mask
                ]
                .mean(
                    axis=0
                )
            )

            corrs = (
                1.0
                - cdist(
                    in_mean,
                    out_mean,
                    metric="correlation",
                )
            )

            corrs = (
                np.nan_to_num(
                    corrs,
                    nan=0.0,
                    posinf=0.0,
                    neginf=0.0,
                )
            )

            res = decoder(
                corrs
            )

            res[
                "rep"
            ] = rep

            res[
                "fold"
            ] = fold

            rows.append(
                res
            )

    return pd.concat(
        rows,
        ignore_index=True,
    )


def summarize_decoding_runs(
    dec_df,
):

    return {
        "mean": float(
            dec_df[
                "accuracy"
            ].mean()
        ),
        "err": float(
            dec_df[
                "accuracy"
            ].std()
            / np.sqrt(
                max(
                    len(
                        dec_df
                    ),
                    1,
                )
            )
        ),
        "std": float(
            dec_df[
                "accuracy"
            ].std()
        ),
        "count": int(
            len(
                dec_df
            )
        ),
        "rank_mean": float(
            dec_df[
                "rank"
            ].mean()
        ),
        "error_mean": float(
            dec_df[
                "error"
            ].mean()
        ),
    }


def decode_subjects(
    subjects,
    analysis_type,
    archetype_indices=None,
    edge_trim=0,
    nfolds=2,
    nreps=20,
    seed=42,
):

    recon_stack = (
        build_recon_stack(
            subjects,
            analysis_type,
            archetype_indices=(
                archetype_indices
            ),
            edge_trim=edge_trim,
        )
    )

    dec_df = (
        run_timepoint_decoding(
            recon_stack,
            nfolds=nfolds,
            nreps=nreps,
            seed=seed,
        )
    )

    return (
        dec_df,
        summarize_decoding_runs(
            dec_df
        ),
    )


## 3. Cross-fitted ranking/evaluation helpers

In [ ]:

def split_outer_subject_indices(
    n_subjects,
    rng,
):

    idx = np.arange(
        n_subjects
    )

    rng.shuffle(
        idx
    )

    midpoint = (
        n_subjects
        // 2
    )

    A = idx[
        :midpoint
    ]

    B = idx[
        midpoint:
    ]

    return A, B


def subset_subjects(
    subjects,
    indices,
):

    return [
        subjects[
            int(i)
        ]
        for i
        in indices
    ]


def rank_archetypes_on_subjects(
    ranking_subjects,
    analysis_type,
    edge_trim,
    nfolds=NFOLDS_DECODE,
    nreps=NREPS_DECODE,
    seed=42,
):

    n_archetypes = int(
        np.asarray(
            ranking_subjects[
                0
            ]["S"]
        ).shape[0]
    )

    rows = []

    for archetype in range(
        n_archetypes
    ):

        _, summary = (
            decode_subjects(
                ranking_subjects,
                analysis_type,
                archetype_indices=[
                    archetype
                ],
                edge_trim=edge_trim,
                nfolds=nfolds,
                nreps=nreps,
                seed=seed,
            )
        )

        rows.append(
            {
                "archetype": int(
                    archetype
                ),
                "mean_accuracy": (
                    summary["mean"]
                ),
                "sem_accuracy": (
                    summary["err"]
                ),
                "rank_mean": (
                    summary[
                        "rank_mean"
                    ]
                ),
                "error_mean": (
                    summary[
                        "error_mean"
                    ]
                ),
            }
        )

    per_arch = pd.DataFrame(
        rows
    )

    ranked = (
        per_arch
        .sort_values(
            [
                "mean_accuracy",
                "rank_mean",
                "archetype",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )[
            "archetype"
        ]
        .astype(int)
        .tolist()
    )

    return (
        ranked,
        per_arch,
    )


def evaluate_top5_on_subjects(
    evaluation_subjects,
    analysis_type,
    ranked_archetypes,
    edge_trim,
    nfolds=NFOLDS_DECODE,
    nreps=NREPS_DECODE,
    seed=42,
):

    _, full_summary = (
        decode_subjects(
            evaluation_subjects,
            analysis_type,
            archetype_indices=None,
            edge_trim=edge_trim,
            nfolds=nfolds,
            nreps=nreps,
            seed=seed,
        )
    )

    selected = (
        ranked_archetypes[
            :TOP_M
        ]
    )

    _, top_summary = (
        decode_subjects(
            evaluation_subjects,
            analysis_type,
            archetype_indices=selected,
            edge_trim=edge_trim,
            nfolds=nfolds,
            nreps=nreps,
            seed=seed,
        )
    )

    example = (
        reconstruct_from_archetypes(
            evaluation_subjects[0],
            analysis_type,
            archetype_indices=selected,
            edge_trim=edge_trim,
        )
    )

    T = example.shape[0]
    chance = 1.0 / T

    denom = (
        full_summary["mean"]
        - chance
    )

    retention = (
        (
            top_summary["mean"]
            - chance
        )
        / denom
        if np.abs(
            denom
        ) > 1e-12
        else np.nan
    )

    return {
        "top_m": TOP_M,
        "selected_archetypes": str(
            selected
        ),
        "mean": top_summary["mean"],
        "err": top_summary["err"],
        "std": top_summary["std"],
        "count": top_summary["count"],
        "rank_mean": (
            top_summary[
                "rank_mean"
            ]
        ),
        "error_mean": (
            top_summary[
                "error_mean"
            ]
        ),
        "full_mean_same_eval": (
            full_summary[
                "mean"
            ]
        ),
        "full_err_same_eval": (
            full_summary[
                "err"
            ]
        ),
        "chance": chance,
        "retention_above_chance": (
            retention
        ),
        "n_timepoints": T,
        "edge_trim": edge_trim,
    }



## 4. Recover the original outer splits

If the original held-out run exists, we read `outer_seed` from its \(K=46\), top-5 rows. This reproduces the exact participant splits used in the original analysis. If those files are unavailable, the notebook generates new outer seeds and later recomputes the untrimmed baseline on those same splits.


In [ ]:

HAVE_BASELINE = (
    BASELINE_RUN_CSV.exists()
    and BASELINE_RANK_CSV.exists()
)

if HAVE_BASELINE:

    baseline_run_all = pd.read_csv(
        BASELINE_RUN_CSV
    )

    baseline_rank_all = pd.read_csv(
        BASELINE_RANK_CSV
    )

    baseline_run_df = (
        baseline_run_all[
            (
                baseline_run_all[
                    "K"
                ] == K
            )
            & (
                baseline_run_all[
                    "top_m"
                ] == TOP_M
            )
            & (
                baseline_run_all[
                    "condition"
                ].isin(
                    CONDITIONS
                )
            )
        ]
        .copy()
    )

    baseline_rank_df = (
        baseline_rank_all[
            (
                baseline_rank_all[
                    "K"
                ] == K
            )
            & (
                baseline_rank_all[
                    "condition"
                ].isin(
                    CONDITIONS
                )
            )
        ]
        .copy()
    )

    seed_df = (
        baseline_run_df[
            [
                "condition",
                "outer_rep",
                "outer_seed",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "condition",
                "outer_rep",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "Using exact outer seeds "
        "from original held-out analysis."
    )

    print(
        "Baseline K=46 top-5 rows:",
        len(
            baseline_run_df
        ),
    )

else:

    print(
        "Original held-out CSVs not found. "
        "Generating fresh paired outer splits."
    )

    rng = np.random.default_rng(
        RNG_SEED
    )

    seed_rows = []

    for condition in CONDITIONS:

        for outer_rep in range(
            N_OUTER_REPS
        ):

            seed_rows.append(
                {
                    "condition": condition,
                    "outer_rep": (
                        outer_rep
                    ),
                    "outer_seed": int(
                        rng.integers(
                            0,
                            2**31 - 1,
                        )
                    ),
                }
            )

    seed_df = pd.DataFrame(
        seed_rows
    )

    baseline_run_df = None
    baseline_rank_df = None


display(
    seed_df.head()
)

print(
    "Seeds per condition:"
)

display(
    seed_df.groupby(
        "condition"
    )[
        "outer_rep"
    ].nunique()
)


## 5. Run the edge-trimmed cross-fit

In [ ]:

trim_run_rows = []
trim_rank_rows = []

for condition in CONDITIONS:

    subjects = (
        get_condition_subjects(
            fit,
            condition,
        )
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"{condition} | "
        f"K={K} | "
        f"N={len(subjects)}"
    )

    print(
        "=" * 80
    )

    condition_seeds = (
        seed_df[
            seed_df[
                "condition"
            ] == condition
        ]
        .sort_values(
            "outer_rep"
        )
    )

    for _, seed_row in (
        condition_seeds
        .iterrows()
    ):

        outer_rep = int(
            seed_row[
                "outer_rep"
            ]
        )

        outer_seed = int(
            seed_row[
                "outer_seed"
            ]
        )

        outer_rng = (
            np.random.default_rng(
                outer_seed
            )
        )

        A_idx, B_idx = (
            split_outer_subject_indices(
                len(
                    subjects
                ),
                outer_rng,
            )
        )

        directions = [
            (
                "A_to_B",
                A_idx,
                B_idx,
            ),
            (
                "B_to_A",
                B_idx,
                A_idx,
            ),
        ]

        for (
            direction,
            ranking_idx,
            evaluation_idx,
        ) in directions:

            ranking_subjects = (
                subset_subjects(
                    subjects,
                    ranking_idx,
                )
            )

            evaluation_subjects = (
                subset_subjects(
                    subjects,
                    evaluation_idx,
                )
            )

            ranking_seed = (
                RNG_SEED
                + 100000
                * outer_rep
                + 1000
                * K
                + (
                    0
                    if direction
                    == "A_to_B"
                    else 1
                )
            )

            evaluation_seed = (
                RNG_SEED
                + 200000
                * outer_rep
                + 1000
                * K
                + (
                    0
                    if direction
                    == "A_to_B"
                    else 1
                )
            )

            ranked, per_arch = (
                rank_archetypes_on_subjects(
                    ranking_subjects,
                    ANALYSIS_TYPE,
                    edge_trim=(
                        EDGE_TRIM
                    ),
                    nfolds=(
                        NFOLDS_DECODE
                    ),
                    nreps=(
                        NREPS_DECODE
                    ),
                    seed=(
                        ranking_seed
                    ),
                )
            )

            per_arch = (
                per_arch.copy()
            )

            per_arch[
                "analysis_type"
            ] = ANALYSIS_TYPE

            per_arch[
                "fit_scope"
            ] = FIT_SCOPE

            per_arch[
                "condition"
            ] = condition

            per_arch[
                "K"
            ] = K

            per_arch[
                "outer_rep"
            ] = outer_rep

            per_arch[
                "direction"
            ] = direction

            per_arch[
                "outer_seed"
            ] = outer_seed

            per_arch[
                "edge_trim"
            ] = EDGE_TRIM

            trim_rank_rows.append(
                per_arch
            )

            heldout = (
                evaluate_top5_on_subjects(
                    evaluation_subjects,
                    ANALYSIS_TYPE,
                    ranked,
                    edge_trim=(
                        EDGE_TRIM
                    ),
                    nfolds=(
                        NFOLDS_DECODE
                    ),
                    nreps=(
                        NREPS_DECODE
                    ),
                    seed=(
                        evaluation_seed
                    ),
                )
            )

            heldout.update(
                {
                    "analysis_type": (
                        ANALYSIS_TYPE
                    ),
                    "fit_scope": (
                        FIT_SCOPE
                    ),
                    "condition": (
                        condition
                    ),
                    "K": K,
                    "outer_rep": (
                        outer_rep
                    ),
                    "direction": (
                        direction
                    ),
                    "outer_seed": (
                        outer_seed
                    ),
                    "n_ranking_subjects": (
                        len(
                            ranking_subjects
                        )
                    ),
                    "n_evaluation_subjects": (
                        len(
                            evaluation_subjects
                        )
                    ),
                }
            )

            trim_run_rows.append(
                heldout
            )

        if (
            outer_rep + 1
        ) % 5 == 0:

            print(
                " completed outer rep "
                f"{outer_rep + 1}/"
                f"{N_OUTER_REPS}"
            )

    # Checkpoint after each condition.
    trim_run_df = (
        pd.DataFrame(
            trim_run_rows
        )
    )

    trim_rank_df = (
        pd.concat(
            trim_rank_rows,
            ignore_index=True,
        )
    )

    trim_run_df.to_csv(
        TRIM_RUN_CSV,
        index=False,
    )

    trim_rank_df.to_csv(
        TRIM_RANK_CSV,
        index=False,
    )

    print(
        " checkpoint saved"
    )


print(
    "\nDone."
)

print(
    "Trimmed held-out rows:",
    len(
        trim_run_df
    ),
)

print(
    "Trimmed ranking rows:",
    len(
        trim_rank_df
    ),
)



## 6. Fallback: recompute the untrimmed baseline only if needed

Normally this cell does nothing because the original held-out CSVs are reused. If those files were missing, it recomputes the \(K=46\), top-5 untrimmed analysis on the **same outer seeds** used above.


In [ ]:

if not HAVE_BASELINE:

    baseline_run_rows = []
    baseline_rank_rows = []

    for condition in CONDITIONS:

        subjects = (
            get_condition_subjects(
                fit,
                condition,
            )
        )

        condition_seeds = (
            seed_df[
                seed_df[
                    "condition"
                ] == condition
            ]
            .sort_values(
                "outer_rep"
            )
        )

        for _, seed_row in (
            condition_seeds
            .iterrows()
        ):

            outer_rep = int(
                seed_row[
                    "outer_rep"
                ]
            )

            outer_seed = int(
                seed_row[
                    "outer_seed"
                ]
            )

            outer_rng = (
                np.random.default_rng(
                    outer_seed
                )
            )

            A_idx, B_idx = (
                split_outer_subject_indices(
                    len(
                        subjects
                    ),
                    outer_rng,
                )
            )

            directions = [
                (
                    "A_to_B",
                    A_idx,
                    B_idx,
                ),
                (
                    "B_to_A",
                    B_idx,
                    A_idx,
                ),
            ]

            for (
                direction,
                ranking_idx,
                evaluation_idx,
            ) in directions:

                ranking_subjects = (
                    subset_subjects(
                        subjects,
                        ranking_idx,
                    )
                )

                evaluation_subjects = (
                    subset_subjects(
                        subjects,
                        evaluation_idx,
                    )
                )

                ranking_seed = (
                    RNG_SEED
                    + 100000
                    * outer_rep
                    + 1000
                    * K
                    + (
                        0
                        if direction
                        == "A_to_B"
                        else 1
                    )
                )

                evaluation_seed = (
                    RNG_SEED
                    + 200000
                    * outer_rep
                    + 1000
                    * K
                    + (
                        0
                        if direction
                        == "A_to_B"
                        else 1
                    )
                )

                ranked, per_arch = (
                    rank_archetypes_on_subjects(
                        ranking_subjects,
                        ANALYSIS_TYPE,
                        edge_trim=0,
                        nfolds=(
                            NFOLDS_DECODE
                        ),
                        nreps=(
                            NREPS_DECODE
                        ),
                        seed=(
                            ranking_seed
                        ),
                    )
                )

                per_arch = (
                    per_arch.copy()
                )

                per_arch[
                    "analysis_type"
                ] = ANALYSIS_TYPE

                per_arch[
                    "fit_scope"
                ] = FIT_SCOPE

                per_arch[
                    "condition"
                ] = condition

                per_arch[
                    "K"
                ] = K

                per_arch[
                    "outer_rep"
                ] = outer_rep

                per_arch[
                    "direction"
                ] = direction

                per_arch[
                    "outer_seed"
                ] = outer_seed

                per_arch[
                    "edge_trim"
                ] = 0

                baseline_rank_rows.append(
                    per_arch
                )

                heldout = (
                    evaluate_top5_on_subjects(
                        evaluation_subjects,
                        ANALYSIS_TYPE,
                        ranked,
                        edge_trim=0,
                        nfolds=(
                            NFOLDS_DECODE
                        ),
                        nreps=(
                            NREPS_DECODE
                        ),
                        seed=(
                            evaluation_seed
                        ),
                    )
                )

                heldout.update(
                    {
                        "analysis_type": (
                            ANALYSIS_TYPE
                        ),
                        "fit_scope": (
                            FIT_SCOPE
                        ),
                        "condition": (
                            condition
                        ),
                        "K": K,
                        "outer_rep": (
                            outer_rep
                        ),
                        "direction": (
                            direction
                        ),
                        "outer_seed": (
                            outer_seed
                        ),
                    }
                )

                baseline_run_rows.append(
                    heldout
                )

    baseline_run_df = (
        pd.DataFrame(
            baseline_run_rows
        )
    )

    baseline_rank_df = (
        pd.concat(
            baseline_rank_rows,
            ignore_index=True,
        )
    )

    baseline_run_df.to_csv(
        OUTPUT_DIR
        / "recomputed_untrimmed_top5_runs.csv",
        index=False,
    )

    baseline_rank_df.to_csv(
        OUTPUT_DIR
        / "recomputed_untrimmed_per_archetype_ranking.csv",
        index=False,
    )

    print(
        "Untrimmed fallback baseline "
        "recomputed."
    )

else:

    print(
        "Using existing original "
        "held-out baseline."
    )



## 7. Primary sensitivity result: baseline vs edge-trimmed top-5

A→B and B→A are two directions of the **same outer participant split**, so they are averaged within `outer_rep` before uncertainty is calculated. The resulting \(n=20\) values per condition are the independent resampling units.

Because trimming changes the number of candidate timepoints, raw chance changes from \(1/300\) to \(1/280\). We therefore report both absolute accuracy and **above-chance accuracy**.


In [ ]:

def direction_average(
    df,
    label,
):

    out = (
        df
        .groupby(
            [
                "condition",
                "outer_rep",
            ],
            as_index=False,
        )
        .agg(
            accuracy=(
                "mean",
                "mean",
            ),
            full_accuracy=(
                "full_mean_same_eval",
                "mean",
            ),
            chance=(
                "chance",
                "mean",
            ),
            retention_full=(
                "retention_above_chance",
                "mean",
            ),
        )
    )

    out[
        "above_chance"
    ] = (
        out[
            "accuracy"
        ]
        - out[
            "chance"
        ]
    )

    out[
        "full_above_chance"
    ] = (
        out[
            "full_accuracy"
        ]
        - out[
            "chance"
        ]
    )

    out[
        "analysis"
    ] = label

    return out


baseline_pair = (
    direction_average(
        baseline_run_df,
        "untrimmed",
    )
)

trim_pair = (
    direction_average(
        trim_run_df,
        f"trim_{EDGE_TRIM}",
    )
)


paired = baseline_pair.merge(
    trim_pair,
    on=[
        "condition",
        "outer_rep",
    ],
    suffixes=(
        "_baseline",
        "_trim",
    ),
)


paired[
    "delta_accuracy"
] = (
    paired[
        "accuracy_trim"
    ]
    - paired[
        "accuracy_baseline"
    ]
)

paired[
    "delta_above_chance"
] = (
    paired[
        "above_chance_trim"
    ]
    - paired[
        "above_chance_baseline"
    ]
)

paired[
    "above_chance_fraction_preserved"
] = (
    paired[
        "above_chance_trim"
    ]
    / paired[
        "above_chance_baseline"
    ]
)


def sem(
    x,
):

    x = pd.Series(
        x
    ).dropna()

    if len(
        x
    ) <= 1:
        return np.nan

    return (
        x.std(
            ddof=1
        )
        / np.sqrt(
            len(
                x
            )
        )
    )


comparison_summary = (
    paired
    .groupby(
        "condition",
        as_index=False,
    )
    .agg(
        baseline_accuracy=(
            "accuracy_baseline",
            "mean",
        ),
        baseline_accuracy_sem=(
            "accuracy_baseline",
            sem,
        ),
        trim_accuracy=(
            "accuracy_trim",
            "mean",
        ),
        trim_accuracy_sem=(
            "accuracy_trim",
            sem,
        ),
        baseline_above_chance=(
            "above_chance_baseline",
            "mean",
        ),
        trim_above_chance=(
            "above_chance_trim",
            "mean",
        ),
        delta_above_chance=(
            "delta_above_chance",
            "mean",
        ),
        fraction_preserved=(
            "above_chance_fraction_preserved",
            "mean",
        ),
        fraction_preserved_sem=(
            "above_chance_fraction_preserved",
            sem,
        ),
        n_outer_reps=(
            "outer_rep",
            "size",
        ),
    )
)


display(
    comparison_summary
)


In [ ]:

# ============================================================
# FIGURE: BASELINE VS TRIMMED
# ============================================================

fig, ax = plt.subplots(
    figsize=(8, 5),
)

x = np.arange(
    len(
        CONDITIONS
    )
)

offset = 0.12

baseline_means = []
baseline_sems = []
trim_means = []
trim_sems = []

for condition in CONDITIONS:

    row = (
        comparison_summary[
            comparison_summary[
                "condition"
            ] == condition
        ]
        .iloc[0]
    )

    baseline_means.append(
        row[
            "baseline_accuracy"
        ]
    )

    baseline_sems.append(
        row[
            "baseline_accuracy_sem"
        ]
    )

    trim_means.append(
        row[
            "trim_accuracy"
        ]
    )

    trim_sems.append(
        row[
            "trim_accuracy_sem"
        ]
    )


ax.errorbar(
    x - offset,
    baseline_means,
    yerr=baseline_sems,
    marker="o",
    capsize=4,
    linewidth=1.8,
    label="Untrimmed",
)

ax.errorbar(
    x + offset,
    trim_means,
    yerr=trim_sems,
    marker="o",
    capsize=4,
    linewidth=1.8,
    label=(
        f"Trim first/last "
        f"{EDGE_TRIM} TRs"
    ),
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    CONDITIONS
)

ax.set_ylabel(
    "Held-out top-5 timepoint identification accuracy"
)

ax.set_title(
    "K=46 edge-trim sensitivity"
)

ax.legend(
    frameon=False
)

fig.tight_layout()

path = (
    OUTPUT_DIR
    / "baseline_vs_edge_trim_top5_accuracy.png"
)

fig.savefig(
    path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    path,
)


In [ ]:

# ============================================================
# FIGURE: ABOVE-CHANCE FRACTION PRESERVED
# ============================================================

fig, ax = plt.subplots(
    figsize=(7, 4.5),
)

plot_df = (
    comparison_summary
    .set_index(
        "condition"
    )
    .reindex(
        CONDITIONS
    )
)

ax.errorbar(
    np.arange(
        len(
            CONDITIONS
        )
    ),
    plot_df[
        "fraction_preserved"
    ],
    yerr=plot_df[
        "fraction_preserved_sem"
    ],
    marker="o",
    capsize=4,
    linewidth=1.8,
)

ax.axhline(
    1.0,
    linestyle=":",
    linewidth=1,
)

ax.set_xticks(
    np.arange(
        len(
            CONDITIONS
        )
    )
)

ax.set_xticklabels(
    CONDITIONS
)

ax.set_ylabel(
    "Trimmed / untrimmed above-chance accuracy"
)

ax.set_title(
    "Fraction of top-5 held-out signal preserved after edge trimming"
)

fig.tight_layout()

path = (
    OUTPUT_DIR
    / "edge_trim_above_chance_fraction_preserved.png"
)

fig.savefig(
    path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    path,
)


## 8. Does trimming change which archetypes are selected?

In [ ]:

GROUP_COLS = [
    "condition",
    "outer_rep",
    "direction",
]


def top5_sets_from_rank_df(
    rank_df,
):

    rows = []

    for keys, sub in (
        rank_df.groupby(
            GROUP_COLS
        )
    ):

        ranked = (
            sub
            .sort_values(
                [
                    "mean_accuracy",
                    "rank_mean",
                    "archetype",
                ],
                ascending=[
                    False,
                    False,
                    True,
                ],
            )[
                "archetype"
            ]
            .astype(int)
            .tolist()
        )

        row = dict(
            zip(
                GROUP_COLS,
                keys,
            )
        )

        row[
            "top5"
        ] = tuple(
            ranked[
                :TOP_M
            ]
        )

        rows.append(
            row
        )

    return pd.DataFrame(
        rows
    )


baseline_sets = (
    top5_sets_from_rank_df(
        baseline_rank_df
    )
)

trim_sets = (
    top5_sets_from_rank_df(
        trim_rank_df
    )
)

set_compare = (
    baseline_sets.merge(
        trim_sets,
        on=GROUP_COLS,
        suffixes=(
            "_baseline",
            "_trim",
        ),
    )
)


def jaccard(
    a,
    b,
):

    a = set(a)
    b = set(b)

    return (
        len(
            a & b
        )
        / len(
            a | b
        )
    )


set_compare[
    "jaccard"
] = [
    jaccard(
        a,
        b,
    )
    for a, b
    in zip(
        set_compare[
            "top5_baseline"
        ],
        set_compare[
            "top5_trim"
        ],
    )
]

set_compare[
    "n_shared"
] = [
    len(
        set(a)
        & set(b)
    )
    for a, b
    in zip(
        set_compare[
            "top5_baseline"
        ],
        set_compare[
            "top5_trim"
        ],
    )
]


jaccard_summary = (
    set_compare
    .groupby(
        "condition",
        as_index=False,
    )
    .agg(
        mean_jaccard=(
            "jaccard",
            "mean",
        ),
        median_jaccard=(
            "jaccard",
            "median",
        ),
        mean_n_shared=(
            "n_shared",
            "mean",
        ),
        median_n_shared=(
            "n_shared",
            "median",
        ),
    )
)


display(
    jaccard_summary
)


In [ ]:

def selection_frequency(
    rank_df,
):

    top_rows = []

    for keys, sub in (
        rank_df.groupby(
            GROUP_COLS
        )
    ):

        ranked = (
            sub
            .sort_values(
                [
                    "mean_accuracy",
                    "rank_mean",
                    "archetype",
                ],
                ascending=[
                    False,
                    False,
                    True,
                ],
            )[
                "archetype"
            ]
            .astype(int)
            .tolist()
        )

        metadata = dict(
            zip(
                GROUP_COLS,
                keys,
            )
        )

        for archetype in (
            ranked[
                :TOP_M
            ]
        ):

            top_rows.append(
                {
                    **metadata,
                    "archetype": (
                        archetype
                    ),
                }
            )

    long_df = pd.DataFrame(
        top_rows
    )

    out = (
        long_df
        .groupby(
            [
                "condition",
                "archetype",
            ],
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size":
                "selected_count"
            }
        )
    )

    n_crossfits = (
        N_OUTER_REPS
        * 2
    )

    out[
        "selection_frequency"
    ] = (
        out[
            "selected_count"
        ]
        / n_crossfits
    )

    return out


base_freq = (
    selection_frequency(
        baseline_rank_df
    )
    .rename(
        columns={
            "selection_frequency":
            "baseline_frequency"
        }
    )
)

trim_freq = (
    selection_frequency(
        trim_rank_df
    )
    .rename(
        columns={
            "selection_frequency":
            "trim_frequency"
        }
    )
)

freq_compare = (
    base_freq[
        [
            "condition",
            "archetype",
            "baseline_frequency",
        ]
    ]
    .merge(
        trim_freq[
            [
                "condition",
                "archetype",
                "trim_frequency",
            ]
        ],
        on=[
            "condition",
            "archetype",
        ],
        how="outer",
    )
    .fillna(
        0.0
    )
)


locked_freq = (
    freq_compare[
        freq_compare[
            "archetype"
        ].isin(
            LOCKED_ARCHETYPES
        )
    ]
    .sort_values(
        [
            "condition",
            "baseline_frequency",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


print(
    "Selection frequencies for "
    "the five interpretation archetypes:"
)

display(
    locked_freq
)


In [ ]:

# ============================================================
# FIGURE: BASELINE VS TRIMMED SELECTION FREQUENCY
# ============================================================

fig, axes = plt.subplots(
    1,
    len(
        CONDITIONS
    ),
    figsize=(15, 4.5),
    sharex=True,
    sharey=True,
)

for ax, condition in zip(
    axes,
    CONDITIONS,
):

    cur = (
        freq_compare[
            freq_compare[
                "condition"
            ] == condition
        ]
    )

    ax.scatter(
        cur[
            "baseline_frequency"
        ],
        cur[
            "trim_frequency"
        ],
        alpha=0.55,
    )

    for archetype in (
        LOCKED_ARCHETYPES
    ):

        row = cur[
            cur[
                "archetype"
            ] == archetype
        ]

        if len(
            row
        ):

            x = float(
                row[
                    "baseline_frequency"
                ].iloc[0]
            )

            y = float(
                row[
                    "trim_frequency"
                ].iloc[0]
            )

            ax.text(
                x,
                y,
                f"A{archetype}",
                fontsize=9,
            )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle=":",
        linewidth=1,
    )

    rho, p = (
        spearmanr(
            cur[
                "baseline_frequency"
            ],
            cur[
                "trim_frequency"
            ],
        )
    )

    ax.set_title(
        f"{condition}\n"
        f"Spearman r={rho:.2f}"
    )

    ax.set_xlabel(
        "Untrimmed selection frequency"
    )

axes[0].set_ylabel(
    "Trimmed selection frequency"
)

fig.suptitle(
    "Top-5 archetype selection stability after edge trimming",
    y=1.03,
)

fig.tight_layout()

path = (
    OUTPUT_DIR
    / "selection_frequency_baseline_vs_trimmed.png"
)

fig.savefig(
    path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    path,
)



## 9. Do the five interpretation archetypes lose their single-archetype signal?

The cross-fitted top-5 analysis above is primary. This final diagnostic looks specifically at the five locked interpretation archetypes and asks whether their **ranking-half single-archetype timepoint-identification performance** collapses when run edges are removed.

This is a sensitivity diagnostic, not a new independent selection procedure.


In [ ]:

def direction_average_rank(
    df,
    value_name,
):

    return (
        df[
            df[
                "archetype"
            ].isin(
                LOCKED_ARCHETYPES
            )
        ]
        .groupby(
            [
                "condition",
                "outer_rep",
                "archetype",
            ],
            as_index=False,
        )
        .agg(
            **{
                value_name: (
                    "mean_accuracy",
                    "mean",
                )
            }
        )
    )


base_arch = (
    direction_average_rank(
        baseline_rank_df,
        "baseline_accuracy",
    )
)

trim_arch = (
    direction_average_rank(
        trim_rank_df,
        "trim_accuracy",
    )
)

arch_compare = (
    base_arch.merge(
        trim_arch,
        on=[
            "condition",
            "outer_rep",
            "archetype",
        ],
    )
)

arch_compare[
    "delta"
] = (
    arch_compare[
        "trim_accuracy"
    ]
    - arch_compare[
        "baseline_accuracy"
    ]
)


arch_summary = (
    arch_compare
    .groupby(
        [
            "condition",
            "archetype",
        ],
        as_index=False,
    )
    .agg(
        baseline_accuracy=(
            "baseline_accuracy",
            "mean",
        ),
        trim_accuracy=(
            "trim_accuracy",
            "mean",
        ),
        mean_delta=(
            "delta",
            "mean",
        ),
        delta_sem=(
            "delta",
            sem,
        ),
    )
)


display(
    arch_summary
)


## 10. Save summary tables

In [ ]:

comparison_summary.to_csv(
    OUTPUT_DIR
    / "edge_trim_accuracy_summary.csv",
    index=False,
)

paired.to_csv(
    OUTPUT_DIR
    / "edge_trim_accuracy_paired_outer_reps.csv",
    index=False,
)

jaccard_summary.to_csv(
    OUTPUT_DIR
    / "edge_trim_top5_jaccard_summary.csv",
    index=False,
)

set_compare.to_csv(
    OUTPUT_DIR
    / "edge_trim_top5_sets_by_crossfit.csv",
    index=False,
)

freq_compare.to_csv(
    OUTPUT_DIR
    / "edge_trim_selection_frequency_comparison.csv",
    index=False,
)

arch_summary.to_csv(
    OUTPUT_DIR
    / "edge_trim_locked_archetype_single_component_summary.csv",
    index=False,
)

print(
    "Saved all summaries to:",
    OUTPUT_DIR.resolve(),
)



## 11. Interpretation guide

The edge-trim sensitivity is reassuring if the following broad pattern holds:

- intact and word top-5 held-out accuracy remain clearly above their new trimmed chance level;
- most of the **above-chance** signal is retained after removing the run edges;
- the top-5 sets continue to overlap substantially across baseline and trimmed ranking;
- the selection frequencies of the locked archetypes remain qualitatively similar;
- no single selected archetype's ranking performance collapses uniquely after trimming.

The purpose is **not** to require numerically identical accuracies. Trimming removes 20 of 300 candidate timepoints and changes the decoding problem itself. The relevant question is whether the sparse held-out result and the identity of the informative motifs remain qualitatively intact when boundary timepoints cannot contribute.

If they do, the manuscript can state that the sparse cross-subject signal was robust to exclusion of the first and last 10 TRs, reducing concern that the selected motifs or held-out decoding were primarily driven by run-boundary structure.
